# CarRacing: public-track mixed fine-tuning of the selected basic CNN

This notebook starts from your own **400,000-transition basic-CNN gSDE checkpoint**, preserves it, and trains an independent copy. It does not use IMPALA or a downloaded pretrained model. Run only after the organizer has allowed public-seed training.

Use 80% ordinary training-pool episode resets and 20% public-track resets, with a smaller learning rate of `1e-4`. This split and rate are experimental choices, not verified optimal settings. The first target is about 10,000 **additional** transitions. Compare all three public tracks and the same ten validation tracks before and after. Protect the original checkpoint and final-test tracks.


## What changes, and what stays protected

The basic CNN, 84x84 grayscale input, four-frame history, action repeat 2, PPO rollout/minibatch settings, original reward and gSDE remain the same. We change the episode sampler and lower the learning rate. Changing two settings together means an improvement would not isolate one causal parameter.

The original folder `basic_cnn/gsde_recovery_01` is read-only to this workflow. Outputs go to `basic_cnn/public_mix_from_400k_01`; first run refuses to overwrite an existing fine-tune checkpoint. The source must report exactly 400,000 transitions by default. The original ZIP is hashed and copied into the new folder. A new candidate is never a reason to delete the baseline.

Public roads become training data in this experiment, so public scores are no longer evidence of generalization. Validation remains outside training; repeated tuning can still overfit validation selection. The final test stays off. If time is short, preserve/export the original rather than assume this fine-tune will help.


## 1. Open in Colab and choose a GPU

1. Open https://colab.research.google.com/ and choose **File → Upload notebook**. Select this file.
2. Choose **Runtime → Change runtime type → Hardware accelerator → GPU**. Select **T4** if available, or another available NVIDIA GPU. Leave runtime language as Python 3. Do not choose TPU for this notebook.
3. Click Save, then Connect. Changing the runtime later can erase the current session memory, so choose the GPU first.
4. Execute cells using the triangle button or Shift+Enter. Read the output before advancing.
5. Preserve the original basic-CNN checkpoint. Run this recovery notebook separately in a new experiment folder; it starts from scratch. Do not train several notebooks simultaneously on one runtime.

GPU type, availability and session duration are not guaranteed. Physics still runs on CPU, so a GPU does not accelerate every part. [Colab FAQ](https://research.google.com/colaboratory/faq.html).


## 2. Install the dependencies

`%pip` installs into this notebook's Python environment. The first line supplies SWIG for Box2D builds. We preserve a compatible preinstalled CUDA-enabled PyTorch; there is no `--upgrade` flag. No system-wide CUDA installation is needed.

Run this installation cell once per new runtime. If Colab asks to restart the session, do so, then continue with the imports below. Installation failure should be resolved before starting training.


In [ ]:
%pip install -q "swig==4.*"
%pip install -q "gymnasium[box2d]==1.3.0" "stable-baselines3==2.9.0" "torch>=2.8,<3.0" "numpy>=1.26,<3" "pillow>=10,<13" "matplotlib>=3.8,<4" "tensorboard>=2.16,<3" "imageio[ffmpeg]>=2.34,<3" "tqdm>=4.66,<5" "rich>=13"


In [ ]:
import os
import shutil
# Headless SDL: Colab has no desktop window. Images/video are displayed inline.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import json
import sys
import time
import importlib.metadata as metadata
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import imageio.v2 as imageio
import gymnasium as gym
import torch
from torch import nn
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack
from IPython.display import Video, display

# Avoid CPU thread oversubscription for these fairly small networks.
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version.split()[0])
print("Gymnasium:", gym.__version__, "SB3:", sb3.__version__, "Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU is fine for setup checks. Select a GPU runtime before long training.")
assert gym.__version__ == "1.3.0", "Use the pinned Gymnasium version for these checks"


## 3. Mount Drive and choose an experiment folder

The notebook file and the training model are different things. Saving the notebook does not save its RAM or model weights. `/content` is temporary; Google Drive keeps checkpoints after a disconnect.

Use `USE_DRIVE=True` in Colab and approve your own Drive mount. Local testing can use `False`. `EXPERIMENT_NAME` identifies one run. Keep its name to resume; choose a new name for a new training seed or changed settings. The CNN and IMPALA notebooks have different architecture folders automatically.


In [ ]:
USE_DRIVE = True
ARCHITECTURE = "basic_cnn"
EXPERIMENT_NAME = "public_mix_from_400k_01"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/max-verstappen")
else:
    OUTPUT_ROOT = Path.cwd() / "racing_outputs"
RUN_DIR = OUTPUT_ROOT / ARCHITECTURE / EXPERIMENT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = Path("/content/racing_logs") / ARCHITECTURE / EXPERIMENT_NAME if USE_DRIVE else RUN_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent models/results:", RUN_DIR)
print("TensorBoard logs for this session:", LOG_DIR)

# This is your selected parent, NOT latest.zip at 500,526 transitions.
BASE_CHECKPOINT = OUTPUT_ROOT / "basic_cnn" / "gsde_recovery_01" / "best_validation.zip"
EXPECTED_BASE_TRANSITIONS = 400_000
PUBLIC_EPISODE_FRACTION = 0.20
if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Selected parent checkpoint missing: {BASE_CHECKPOINT}")
print("Parent checkpoint:", BASE_CHECKPOINT)


## 4. Fine-tuning settings and separate track groups

`TRAINING_SEEDS` remains the ordinary 100-road pool, 1000-1099. The new sampler additionally selects one of 202, 42 and 203 on 20% of training episode resets. The branch probability is per reset, not a promise of exactly 20% of transitions or balanced short chunks. Validation 10000-10019 and test 20000-20019 are never sampled for training.

The learning rate changes from `3e-4` to `1e-4`; the model's actual schedule is refreshed after loading, rather than only editing CONFIG. Rollouts remain 2 x 512 decisions, minibatches 128 and up to five update passes. gSDE, four-decision noise resampling, log noise initialization metadata, zero entropy bonus, discount, GAE, clip range and critic weight are preserved. Loading existing scales does not reinitialize them from the log_std_init setting.

The unchanged base pool is saved in CONFIG; the complete mixed sampling recipe is separately saved in `sampling.json` and the checkpoint's `training_sampling` metadata.


In [ ]:
TRAINING_SEEDS = tuple(range(1000, 1100))
VALIDATION_SEEDS = tuple(range(10000, 10020))
TEST_SEEDS = tuple(range(20000, 20020))
PUBLIC_SEEDS = (202, 42, 203)
assert not set(TRAINING_SEEDS) & set(VALIDATION_SEEDS + TEST_SEEDS + PUBLIC_SEEDS)

CONFIG = {
    "architecture": ARCHITECTURE,
    "run_seed": 7,
    "training_seeds": list(TRAINING_SEEDS),
    "image_size": 84, "frame_stack": 4, "action_repeat": 2,
    "n_envs": 2, "n_steps": 512, "batch_size": 128, "n_epochs": 5,
    "learning_rate": 1e-4, "gamma": 0.99, "gae_lambda": 0.95,
    "clip_range": 0.2, "ent_coef": 0.0, "vf_coef": 0.5,
    "use_sde": True, "sde_sample_freq": 4, "log_std_init": -2.0,
    "max_grad_norm": 0.5, "target_kl": 0.03,
    "features_dim": 256,
}
assert CONFIG["n_steps"] * CONFIG["n_envs"] % CONFIG["batch_size"] == 0

config_path = RUN_DIR / "config.json"
if config_path.exists():
    previous_config = json.loads(config_path.read_text())
    if previous_config != CONFIG:
        raise ValueError("Settings differ from this folder's run. Choose a NEW EXPERIMENT_NAME.")
config_path.write_text(json.dumps(CONFIG, indent=2))
versions = "\n".join(sorted(f"{dist.metadata['Name']}=={dist.version}" for dist in metadata.distributions())) + "\n"
(RUN_DIR / "requirements_runtime.txt").write_text(versions)
print(json.dumps(CONFIG, indent=2))

if not 0 <= PUBLIC_EPISODE_FRACTION <= 1:
    raise ValueError("PUBLIC_EPISODE_FRACTION must be between zero and one")
SAMPLING = {
    "type": "public_episode_mix", "public_probability": PUBLIC_EPISODE_FRACTION,
    "ordinary_seeds": list(TRAINING_SEEDS), "public_seeds": list(PUBLIC_SEEDS),
}
sampling_path = RUN_DIR / "sampling.json"
if sampling_path.exists() and json.loads(sampling_path.read_text()) != SAMPLING:
    raise ValueError("Sampling changed; choose a new fine-tune experiment folder")
sampling_path.write_text(json.dumps(SAMPLING, indent=2))


## 5. Inspect what the environment really returns

The agent controls `[steering, gas, brake]`. Steering ranges from -1 to 1; gas and brake from 0 to 1. The picture has 96 rows, 96 columns and three color channels. `uint8` stores integer pixel intensities from 0 to 255; it is not a neural-network type.

`reset(seed=...)` starts a track. `step(action)` advances the simulator and returns a new image, reward, `terminated`, `truncated`, and metadata. Termination can mean a finish or a failure. A time limit is truncation. Neither reset nor step trains a model.

This preview uses one training track, not a held-out test track. It closes its environment afterward. Using `render_mode="human"` would try to open a desktop window, which is not the Colab display workflow.


In [ ]:
preview = gym.make("CarRacing-v3", continuous=True, max_episode_steps=3000)
try:
    image, info = preview.reset(seed=TRAINING_SEEDS[0])
    print("Observation:", image.shape, image.dtype)
    print("Action bounds:", preview.action_space.low, preview.action_space.high)
    assert image.shape == (96, 96, 3) and image.dtype == np.uint8
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title("Raw image: the policy's source of information")
    plt.axis("off")
    plt.show()
finally:
    preview.close()


## 6. Build preprocessing and track sampling

The wrapper order is deliberate:

`CarRacing + 3000-step TimeLimit → track sampler → repeat action → grayscale/resize → Monitor → vector batch → transpose → frame stack`.

`TrackSampler` chooses training tracks randomly or evaluation tracks in a fixed order. Seeds are logging metadata, never policy inputs. Our explicit pool controls separation; it is different from passing a track seed to the base environment directly.

`ActionRepeat` sends one decision to at most two simulator steps, sums their rewards and stops immediately at an episode boundary. Its counter gives actual simulator steps for evaluation. The inner time limit still allows only 3000 raw steps.

`GrayResize` converts to grayscale and 84×84 while preserving uint8 and a singleton channel. SB3 later divides by 255, exactly once. No extra normalization belongs in these wrappers.

`VecTransposeImage` puts channels first for PyTorch. `VecFrameStack` builds `(n_envs, 4, 84, 84)`. SB3 zero-pads missing history at reset and clears history between episodes. With repeat 2, consecutive stacked frames are two simulator steps apart.

SB3's vector API differs from Gymnasium: reset returns just observations; step returns `(observations, rewards, dones, infos)` and automatically resets finished episodes. Terminal images remain in `info['terminal_observation']`, which SB3 uses for time-limit bootstrapping.


In [ ]:
class TrackSampler(gym.Wrapper):
    """Choose tracks from one explicit pool; the policy never receives the seed."""
    def __init__(self, env, seeds, random_order=True, sampler_seed=7, public_probability=0.0):
        super().__init__(env)
        self.seeds = tuple(int(seed) for seed in seeds)
        if not self.seeds:
            raise ValueError("The track pool cannot be empty")
        self.random_order = random_order
        if not 0 <= public_probability <= 1:
            raise ValueError("Invalid public episode probability")
        self.public_probability = public_probability
        self.track_group = "ordinary"
        self.rng = np.random.default_rng(sampler_seed)
        self.index = 0
        self.track_seed = None
        self.exposure_counts = {}  # Counts episode starts, not finishes or transitions.

    def reset(self, *, seed=None, options=None):
        # Here seed controls the SAMPLER. The chosen track seed is passed on
        # to CarRacing. SB3 seeds each worker once before its first reset.
        if seed is not None:
            self.rng = np.random.default_rng(seed)
            self.index = 0
        if self.random_order:
            use_public = self.public_probability > 0 and self.rng.random() < self.public_probability
            pool = PUBLIC_SEEDS if use_public else self.seeds
            self.track_group = "public" if use_public else "ordinary"
            self.track_seed = int(self.rng.choice(pool))
        else:
            self.track_seed = self.seeds[self.index % len(self.seeds)]
            self.index += 1
        observation, info = self.env.reset(seed=self.track_seed, options=options)
        self.exposure_counts[self.track_seed] = self.exposure_counts.get(self.track_seed, 0) + 1
        return observation, {**info, "track_seed": self.track_seed, "track_group": self.track_group}

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        return observation, reward, terminated, truncated, {
            **info, "track_seed": self.track_seed, "track_group": self.track_group,
        }


class ActionRepeat(gym.Wrapper):
    """Repeat a decision, sum rewards, and count actual simulator steps."""
    def __init__(self, env, repeat=2):
        super().__init__(env)
        if repeat < 1:
            raise ValueError("repeat must be positive")
        self.repeat = repeat
        self.raw_steps = 0

    def reset(self, **kwargs):
        self.raw_steps = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        total_reward = 0.0
        for _ in range(self.repeat):
            observation, reward, terminated, truncated, info = self.env.step(action)
            self.raw_steps += 1
            total_reward += float(reward)
            # Never continue stepping after either kind of episode ending.
            if terminated or truncated:
                break
        return observation, total_reward, terminated, truncated, {
            **info, "raw_steps": self.raw_steps,
        }


class GrayResize(gym.ObservationWrapper):
    """RGB -> resized grayscale; keep uint8 until SB3 normalizes it once."""
    def __init__(self, env, size=84):
        super().__init__(env)
        self.size = size
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(size, size, 1), dtype=np.uint8,
        )

    def observation(self, observation):
        image = Image.fromarray(observation).convert("L")
        image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.uint8)[..., None].copy()


def make_single_env(seeds, random_order, sampler_seed, render_mode=None):
    # TimeLimit is INSIDE action repetition: 3000 means raw simulator steps.
    env = gym.make(
        "CarRacing-v3", continuous=True, max_episode_steps=3000,
        render_mode=render_mode,
    )
    env = TrackSampler(
        env, seeds, random_order, sampler_seed,
        public_probability=PUBLIC_EPISODE_FRACTION if random_order and tuple(seeds) == TRAINING_SEEDS else 0.0,
    )
    env = ActionRepeat(env, repeat=CONFIG["action_repeat"])
    env = GrayResize(env, size=CONFIG["image_size"])
    return Monitor(env)


def build_vec_env(seeds, n_envs=1, random_order=True, render_mode=None):
    factories = []
    for rank in range(n_envs):
        # Bind rank now, rather than letting every factory use the last rank.
        def factory(rank=rank):
            return make_single_env(
                seeds, random_order, CONFIG["run_seed"] + rank, render_mode,
            )
        factories.append(factory)
    # DummyVecEnv runs its workers sequentially on the CPU. It batches their
    # observations for the network; it does not make the physics run on a GPU.
    env = DummyVecEnv(factories)
    env = VecTransposeImage(env)  # (84,84,1) -> (1,84,84)
    env = VecFrameStack(env, n_stack=CONFIG["frame_stack"], channels_order="first")
    env.seed(CONFIG["run_seed"])
    return env


def training_exposure():
    """Combine episode-start counts across the vector workers."""
    counts = {}
    for worker in train_env.get_attr("exposure_counts"):
        for seed, count in worker.items():
            counts[int(seed)] = counts.get(int(seed), 0) + int(count)
    return counts


In [ ]:
# Inspect the policy boundary before training anything.
probe = build_vec_env(TRAINING_SEEDS, n_envs=2)
try:
    stacked = probe.reset()
    assert stacked.shape == (2, 4, 84, 84)
    assert stacked.dtype == np.uint8
    assert np.all(stacked[:, :3] == 0), "Missing history should be zero at reset"
    actions = np.array([[0.0, 0.2, 0.0]] * 2, dtype=np.float32)
    next_stacked, rewards, dones, infos = probe.step(actions)
    assert all(item["raw_steps"] == 2 for item in infos)
    assert all(item["track_seed"] in TRAINING_SEEDS + PUBLIC_SEEDS for item in infos)
    print("Shape/dtype:", stacked.shape, stacked.dtype)
    print("Sampled tracks:", [item["track_seed"] for item in infos])
    print("Reward after one decision:", rewards)
    fig, axes = plt.subplots(1, 4, figsize=(10, 3))
    for i, axis in enumerate(axes):
        axis.imshow(next_stacked[0, i], cmap="gray", vmin=0, vmax=255)
        axis.set_title(f"History slot {i}")
        axis.axis("off")
    plt.show()
finally:
    probe.close()


## 7. Implement the Basic CNN

A convolution slides learned filters across the image. Channels are learned feature maps; they are not necessarily human-readable road detectors. ReLU applies `max(0, x)` so the network can learn nonlinear relationships. Flatten changes the layout, not the learned meaning. The final linear layer produces 256 features for the actor and critic.

The three convolution outputs are `(32,20,20)`, `(64,9,9)`, and `(64,7,7)`. Flatten produces 3136 numbers, then a linear layer maps them to 256. This follows the classic Nature-CNN layout with a smaller 256-feature projection.

The dummy tensor in `__init__` determines the flatten size without hard-coding it. `torch.no_grad()` avoids building a gradient graph for that shape calculation. `forward()` defines the actual image-to-feature computation. SB3's preprocessing handles normalization before calling it.

Start here to establish a baseline before increasing complexity.


In [ ]:
class BasicCNN(BaseFeaturesExtractor):
    """Three convolutional layers followed by a 256-number feature vector."""
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        channels = observation_space.shape[0]
        self.convolutions = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, *observation_space.shape)
            flattened_size = self.convolutions(dummy).shape[1]
        self.projection = nn.Sequential(
            nn.Linear(flattened_size, features_dim), nn.ReLU(),
        )

    def forward(self, observations):
        # SB3 already converts uint8 pixels to float32 / 255 before this call.
        return self.projection(self.convolutions(observations))


Encoder = BasicCNN


In [ ]:
observation_space = gym.spaces.Box(0, 255, shape=(4, 84, 84), dtype=np.uint8)
encoder = Encoder(observation_space, features_dim=CONFIG["features_dim"])
with torch.no_grad():
    features = encoder(torch.zeros(2, 4, 84, 84))
assert features.shape == (2, 256)
print(encoder)
print("Feature shape:", features.shape)
print("Encoder parameters:", sum(p.numel() for p in encoder.parameters()))
del encoder


## 8. Understand PPO's actor, critic and action distribution

The shared CNN supplies 256 features to two separate MLP heads. Each head has a 64-unit Tanh hidden layer. The actor produces three Gaussian means. gSDE learns a matrix of noise scales over the 64 actor features and three controls. It samples an exploration matrix and reuses it for four vector steps, while changes in the state features still change the resulting action noise. This provides more temporal structure than drawing independent action noise at every decision. The effective action standard deviation depends on the current features: `exp(-2)` is a noise-weight scale here, not a fixed standard deviation for each control. The critic produces one value estimate per observation. A stochastic Gaussian may sample outside the legal action range: SB3 stores the original sample/log probability for PPO and clips the action sent to the game. Deterministic prediction uses the mean and the same bounds. We do not add a new tanh action transformation at evaluation.

**What SB3 handles internally:** collect a fresh rollout with the old policy; compute GAE and return targets; normalize advantages; perform shuffled minibatch updates using the clipped policy loss, value loss and entropy term; clip gradients; collect fresh experience. Old values, probabilities and targets are fixed during each update. This notebook does not claim to implement those internals from scratch.

GAE starts from `delta = reward + gamma × next_value − value`. It combines later errors with `gamma × lambda`, but does not propagate advantages across episode boundaries. True termination has zero future value. Time-limit truncation normally bootstraps from the final observation, not the auto-reset image. Ending a rollout midway through a race also needs a next-value estimate.

The actor objective is `mean(min(ratio × advantage, clip(ratio, 1-epsilon, 1+epsilon) × advantage))`, where `ratio = exp(new_log_prob − old_log_prob)`. We minimize its negative. Clipping limits the incentive for certain probability-ratio changes; it is not a guarantee of safe driving or bounded total network change.


In [ ]:
# A numerical illustration only: this cell does NOT update the real policy.
old_log_prob = torch.tensor([-1.0, -1.0])
new_log_prob = old_log_prob + torch.log(torch.tensor([1.4, 0.6]))
advantage = torch.tensor([2.0, -2.0])
ratio = (new_log_prob - old_log_prob).exp()
unclipped = ratio * advantage
clipped = ratio.clamp(0.8, 1.2) * advantage
actor_loss = -torch.minimum(unclipped, clipped).mean()
print("Ratios:", ratio.tolist())
print("Unclipped terms:", unclipped.tolist())
print("Clipped terms:", clipped.tolist())
print("Actor loss:", actor_loss.item())
# For both examples the favorable ratio change is already beyond clipping.
# This example illustrates the objective, not a complete optimizer update.


## 9. Load two independent models: protected parent and fine-tune copy

For the first fine-tune leave `RESUME_FROM=""`: this cell loads the selected parent twice, once for baseline evaluation and once with a new mixed training environment. It never initializes a new random policy. After restarting, set `RESUME_FROM=str(RUN_DIR / "latest.zip")` to resume the fine-tune, not to restart from 400k again.

The source configuration must match except for the intentionally reduced learning rate. The checkpoint, its hash and its original timestep are recorded. A changed parent is rejected in an existing fine-tune folder. The loaded actor, critic, learned exploration scales and optimizer are preserved. Only the learning-rate schedule is intentionally changed. Simulator/RNG state is not restored exactly.


In [ ]:
import hashlib

RESUME_FROM = ""  # After a restart: str(RUN_DIR / "latest.zip")

base_model = PPO.load(str(BASE_CHECKPOINT), device=DEVICE)
BASE_STEPS = base_model.num_timesteps
if BASE_STEPS != EXPECTED_BASE_TRANSITIONS:
    raise ValueError(f"Expected selected parent at {EXPECTED_BASE_TRANSITIONS}, found {BASE_STEPS}")
parent_config = getattr(base_model, "experiment_config", None)
if not isinstance(parent_config, dict):
    raise ValueError("Parent lacks experiment metadata")
for key, value in CONFIG.items():
    if key != "learning_rate" and parent_config.get(key) != value:
        raise ValueError(f"Parent setting does not match: {key}")
PARENT = {
    "path": str(BASE_CHECKPOINT), "timesteps": BASE_STEPS,
    "sha256": hashlib.sha256(BASE_CHECKPOINT.read_bytes()).hexdigest(),
    "original_config": parent_config,
}
parent_record = RUN_DIR / "parent.json"
if parent_record.exists() and json.loads(parent_record.read_text()) != PARENT:
    raise ValueError("Parent identity changed; use a new fine-tune folder")
parent_record.write_text(json.dumps(PARENT, indent=2))
parent_copy = RUN_DIR / f"parent_{BASE_STEPS}.zip"
if parent_copy.exists():
    if hashlib.sha256(parent_copy.read_bytes()).hexdigest() != PARENT["sha256"]:
        raise ValueError("Preserved parent differs from the source")
else:
    shutil.copy2(BASE_CHECKPOINT, parent_copy)

if "train_env" in globals():
    train_env.close()
train_env = build_vec_env(TRAINING_SEEDS, n_envs=CONFIG["n_envs"])
if RESUME_FROM:
    model = PPO.load(RESUME_FROM, env=train_env, device=DEVICE)
    if model.experiment_config != CONFIG or getattr(model, "training_sampling", None) != SAMPLING:
        raise ValueError("Fine-tune checkpoint settings/sampling differ")
    if getattr(model, "finetune_parent", None) != PARENT:
        raise ValueError("Fine-tune checkpoint belongs to another parent")
else:
    if (RUN_DIR / "latest.zip").exists():
        raise FileExistsError("Fine-tune exists: set RESUME_FROM instead of restarting")
    model = PPO.load(str(BASE_CHECKPOINT), env=train_env, device=DEVICE)
    model.learning_rate = CONFIG["learning_rate"]
    model._setup_lr_schedule()  # Editing CONFIG alone would leave the old schedule active.
    model.experiment_config = CONFIG.copy()
    model.training_sampling = SAMPLING.copy()
    model.finetune_parent = PARENT.copy()
# Use fresh logs in this experiment, never the parent's saved logging path.
from stable_baselines3.common.logger import configure
model.tensorboard_log = str(LOG_DIR)
model.set_logger(configure(str(LOG_DIR / f"segment_{model.num_timesteps}_{time.time_ns()}"),
                           ["stdout", "csv", "tensorboard"]))
print("Protected parent:", BASE_STEPS)
print("Fine-tune starts/resumes at:", model.num_timesteps)
print("Device:", model.device)
print("Actual scheduled learning rate:", model.lr_schedule(1.0))
print("Output folder:", RUN_DIR)

# Compare under the same runtime/device; a restart onto CPU needs a new baseline.
EVALUATION_CONTEXT = {
    "python": sys.version.split()[0], "gymnasium": gym.__version__,
    "sb3": sb3.__version__, "torch": str(torch.__version__),
    "numpy": np.__version__, "pillow": metadata.version("pillow"),
    "device": str(base_model.device), "threads": torch.get_num_threads(),
}
context_key = hashlib.sha256(json.dumps(EVALUATION_CONTEXT, sort_keys=True).encode()).hexdigest()[:12]
BASELINE_PATH = RUN_DIR / f"baseline_{context_key}.json"
model.finetune_eval_context = EVALUATION_CONTEXT.copy()


## 10. Evaluate actual finishes and save useful checkpoints

`evaluate_tracks` creates separate environments with fixed track seeds. It uses deterministic actions and never calls `learn()`. It reads only completion/timing metadata for scoring; the policy still receives images only.

A real finish is detected through Gymnasium 1.3.0's `info['lap_finished']`, not merely `done=True`. We count the raw steps from our action-repeat wrapper and divide by 50 simulator steps per second. A finish uses its time; a failure/time-out receives 70 seconds. These are local challenge-style measurements, to be checked against the organizer's exact evaluator. No hidden coordinates or track geometry are read.

`RacingCallback` saves `latest.zip` every 25,000 policy transitions and checks five fixed validation tracks every 100,000. `best_validation.zip` minimizes mean penalized time, breaking equal-time ties using higher time-adjusted raw reward (`reward + 0.1 × raw_steps`) and retaining the earlier checkpoint when both are equal. The tie-breaker uses the original reward and retains any off-map penalty; it is a progress proxy, not an exact tile count. This avoids preferring an early crash just because it accrued fewer time penalties. If every checkpoint fails all tracks, they tie at 70 seconds: that means there is still no evidence of successful lap completion, even if the progress proxy improves. The 20-track validation cell later provides a broader audit.

SB3's vector environment auto-resets after a finish. The evaluator uses the ending episode's metadata from `infos`, rather than counting the new episode. Optional video rendering is streamed to disk and does not affect the actor's inputs.


In [ ]:
def evaluate_tracks(policy, seeds, split="validation", video_path=None):
    """Evaluate fixed tracks without updating weights; optionally stream video."""
    pools = {
        "validation": VALIDATION_SEEDS, "test": TEST_SEEDS,
        "public": PUBLIC_SEEDS, "training": TRAINING_SEEDS,
    }
    if split not in pools or not set(seeds).issubset(set(pools[split])):
        raise ValueError("Evaluation seeds must belong to the requested split")
    if video_path is not None and len(seeds) != 1:
        raise ValueError("Record one track at a time")
    rows = []
    for seed in seeds:
        env = build_vec_env(
            (seed,), n_envs=1, random_order=False,
            render_mode="rgb_array" if video_path else None,
        )
        writer = None
        try:
            observation = env.reset()
            total_reward = 0.0
            if video_path:
                # Stream to disk rather than accumulating hundreds of MB in RAM.
                writer = imageio.get_writer(
                    str(video_path), fps=50 / CONFIG["action_repeat"],
                )
            while True:
                action, _ = policy.predict(observation, deterministic=True)
                observation, rewards, dones, infos = env.step(action)
                total_reward += float(rewards[0])
                if writer and not dones[0]:
                    frame = env.envs[0].render()
                    frame = Image.fromarray(frame).resize((480, 320))
                    writer.append_data(np.asarray(frame))
                if dones[0]:
                    info = infos[0]  # Metadata belongs to the episode that ended.
                    # SB3 auto-resets: observation is now the NEW episode's image.
                    # The final image, if needed, is info['terminal_observation'].
                    if not info.get("TimeLimit.truncated", False) and "lap_finished" not in info:
                        raise RuntimeError("This Gymnasium version lacks lap completion metadata")
                    finished = bool(info.get("lap_finished", False))
                    steps = int(info["raw_steps"])
                    lap_seconds = steps / 50.0 if finished else None
                    rows.append({
                        "split": split, "track_seed": int(seed),
                        "finished": finished, "raw_steps": steps,
                        "lap_seconds": lap_seconds,
                        "penalized_seconds": lap_seconds if finished else 70.0,
                        "reward": total_reward,
                    })
                    break
        finally:
            if writer is not None:
                writer.close()
            env.close()
    return rows


def summarize(rows):
    finishes = [row["lap_seconds"] for row in rows if row["finished"]]
    return {
        "tracks": len(rows),
        "completion_rate": sum(row["finished"] for row in rows) / len(rows),
        "mean_penalized_seconds": float(np.mean([row["penalized_seconds"] for row in rows])),
        "mean_finishing_seconds": float(np.mean(finishes)) if finishes else None,
        "mean_reward": float(np.mean([row["reward"] for row in rows])),
    }


def save_checkpoint(policy, name):
    """Write a complete SB3 model, including optimizer state, to Drive/local disk."""
    final_path = RUN_DIR / f"{name}.zip"
    temporary_path = RUN_DIR / f".{name}.tmp.zip"
    policy.save(str(temporary_path))
    os.replace(temporary_path, final_path)
    return final_path


class RacingCallback(BaseCallback):
    """Checkpoint and validate by policy-transition count, not callback calls."""
    def __init__(self, save_every=25000, evaluate_every=100000):
        super().__init__()
        self.save_every = save_every
        self.evaluate_every = evaluate_every
        self.next_save = None
        self.next_evaluation = None
        self.history_path = RUN_DIR / "validation_history.json"
        self.history = json.loads(self.history_path.read_text()) if self.history_path.exists() else []
        # All-DNF checkpoints tie at 70 seconds. For ties, compare original
        # reward with its elapsed-time penalty added back. Raw reward alone
        # could favor crashing early simply because fewer time penalties accrue.
        self.best_key = min(
            (self.selection_key(item["summary"], item["tracks"]) for item in self.history),
            default=(float("inf"), float("inf")),
        )

    @staticmethod
    def selection_key(summary, rows):
        # This proxy retains the environment's off-map penalty. It is NOT an
        # exact tile count and assumes unchanged CarRacing reward (-0.1/raw step).
        progress_proxy = float(np.mean([
            row["reward"] + 0.1 * row["raw_steps"] for row in rows
        ]))
        return (summary["mean_penalized_seconds"], -progress_proxy)

    def _on_training_start(self):
        steps = self.model.num_timesteps
        self.next_save = (steps // self.save_every + 1) * self.save_every
        self.next_evaluation = (steps // self.evaluate_every + 1) * self.evaluate_every

    def _on_step(self):
        # num_timesteps increases by n_envs at each vector-environment step.
        steps = self.model.num_timesteps
        if steps >= self.next_save:
            save_checkpoint(self.model, "latest")
            self.next_save += self.save_every
        if steps >= self.next_evaluation:
            # Five fixed tracks keep frequent evaluation affordable.
            rows = evaluate_tracks(self.model, VALIDATION_SEEDS[:5])
            summary = summarize(rows)
            item = {"timesteps": steps, "summary": summary, "tracks": rows}
            self.history.append(item)
            self.history_path.write_text(json.dumps(self.history, indent=2))
            score = summary["mean_penalized_seconds"]
            self.logger.record("validation/penalized_seconds", score)
            self.logger.record("validation/completion_rate", summary["completion_rate"])
            print(f"\nValidation at {steps:,} transitions: {summary}")
            selection_key = self.selection_key(summary, rows)
            if selection_key < self.best_key:
                self.best_key = selection_key
                save_checkpoint(self.model, "best_validation")
                (RUN_DIR / "best_validation_score.json").write_text(json.dumps(item, indent=2))
            self.next_evaluation += self.evaluate_every
        return True


## 10a. Measure the protected baseline before changing weights

Run this once: three public races and ten validation races. It does not learn. Results are cached in the fine-tune folder and tied to the parent hash plus package versions and device. A changed runtime gets its own baseline report. This records public performance before public training. It can take several minutes; do not skip it and assume later scores improved.


In [ ]:
RUN_BASELINE_EVALUATION = True
baseline_path = BASELINE_PATH
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text())
    if baseline["parent"] != PARENT or baseline["context"] != EVALUATION_CONTEXT:
        raise ValueError("Cached baseline belongs to another source checkpoint")
    print("Using preserved baseline:", baseline_path)
elif RUN_BASELINE_EVALUATION:
    public_rows = evaluate_tracks(base_model, PUBLIC_SEEDS, split="public")
    print("Baseline public:", summarize(public_rows), flush=True)
    validation_rows = evaluate_tracks(base_model, VALIDATION_SEEDS[:10], split="validation")
    baseline = {
        "parent": PARENT, "context": EVALUATION_CONTEXT, "public": summarize(public_rows),
        "validation": summarize(validation_rows), "tracks": public_rows + validation_rows,
    }
    baseline_path.write_text(json.dumps(baseline, indent=2))
    print("Baseline validation:", baseline["validation"])
else:
    print("Baseline not measured; long fine-tuning remains blocked.")


## 11. Optional smoke update

The parent is already trained. Leave `RUN_SMOKE=False` unless debugging the new environment. A smoke update is additional training and is counted toward the target. A parent backup is already preserved. The protected baseline evaluation above must finish before long training.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    started = time.perf_counter()
    initial_steps = model.num_timesteps
    model.learn(
        total_timesteps=CONFIG["n_steps"] * CONFIG["n_envs"],
        reset_num_timesteps=False, tb_log_name="ppo", progress_bar=False,
    )
    elapsed = time.perf_counter() - started
    added_steps = model.num_timesteps - initial_steps
    print(f"Added {added_steps:,} policy transitions in {elapsed:.1f}s")
    print(f"Rough throughput: {added_steps / elapsed:.1f} transitions/s")
    print(f"At that speed, 1M transitions: {1_000_000 / (added_steps / elapsed) / 3600:.2f} hours, before extra evaluation")
    checkpoint = save_checkpoint(model, "latest")
    check_env = build_vec_env((VALIDATION_SEEDS[0],), random_order=False)
    try:
        check_obs = check_env.reset()
        before, _ = model.predict(check_obs, deterministic=True)
        loaded = PPO.load(str(checkpoint), device=DEVICE)
        after, _ = loaded.predict(check_obs, deterministic=True)
        np.testing.assert_allclose(before, after, rtol=1e-5, atol=1e-6)
        assert np.isfinite(after).all()
        print("Saved/reloaded actions agree:", after)
        del loaded
    finally:
        check_env.close()


## 12. Run about 10k additional transitions on the copy

Only after the baseline report exists, set `RUN_LONG_TRAINING=True` and run this cell. `ADDITIONAL_TIMESTEPS=10_000` makes the cumulative target parent+10k, around 410k rather than 500k. PPO rounds to complete a rollout. Resuming the same target only finishes its remaining budget. Changing to 20k means parent+20k total, not another 20k on top.

Keep the original CNN, rollout/minibatch sizes and `progress_bar=False`. Checkpoints are saved every 5k; periodic validation is every 10k. The protected parent is outside selection and cannot be overwritten. If the runtime hard-disconnects, resume the fine-tune's latest file in this folder. Allow time for after-training evaluation and submission packaging; no improvement is guaranteed.

The training cell reports how often each public road was selected during this segment. A 20% probability at episode reset does not guarantee all three roads appear in a short run, and it does not mean 20% of transitions. These counts record starts, including automatic resets, rather than completed races. A hard runtime disconnection can prevent the final exposure report from being written.


In [ ]:
# Use ordinary SB3 text logs: Rich progress-bar output recursed in Colab at validation.
RUN_LONG_TRAINING = False  # Change to True after inspecting the short check.
ADDITIONAL_TIMESTEPS = 10_000  # First experiment, NOT another 500k.
TARGET_TOTAL_TIMESTEPS = BASE_STEPS + ADDITIONAL_TIMESTEPS

if RUN_LONG_TRAINING:
    if not BASELINE_PATH.exists():
        raise RuntimeError("Run the protected baseline evaluation first")
    if DEVICE != "cuda":
        print("Running on CPU; measure throughput before committing to a large budget.")
    remaining = max(0, TARGET_TOTAL_TIMESTEPS - model.num_timesteps)
    if remaining:
        callback = RacingCallback(save_every=5_000, evaluate_every=10_000)
        started = time.perf_counter()
        start_steps = model.num_timesteps
        exposure_before = training_exposure()
        try:
            model.learn(
                total_timesteps=remaining, callback=callback,
                reset_num_timesteps=False, tb_log_name="ppo", progress_bar=False,
            )
        finally:
            saved = save_checkpoint(model, "latest")
            exposure_after = training_exposure()
            exposure_delta = {seed: count - exposure_before.get(seed, 0)
                              for seed, count in exposure_after.items()}
            exposure_report = {
                "start_transitions": start_steps, "end_transitions": model.num_timesteps,
                "episode_starts_by_seed": exposure_delta,
                "public_episode_starts": {seed: exposure_delta.get(seed, 0) for seed in PUBLIC_SEEDS},
                "note": "Counts resets during this segment, including automatic resets. A start need not finish or receive an action. Not transition proportions.",
            }
            exposure_path = RUN_DIR / f"exposure_{start_steps}_{model.num_timesteps}.json"
            exposure_path.write_text(json.dumps(exposure_report, indent=2))
            print("Public episode starts:", exposure_report["public_episode_starts"])
            print("Exposure report:", exposure_path)
            elapsed = time.perf_counter() - started
            completed = model.num_timesteps - start_steps
            print("Checkpoint:", saved)
            print(f"This segment: {completed:,} transitions, {elapsed / 60:.1f} minutes")
            if completed:
                print(f"Measured segment throughput (including callbacks): {completed / elapsed:.1f}/s")
    else:
        print("Target already reached. Increase the cumulative target to continue.")
else:
    print("Long training is off. Inspect the checks, then enable it explicitly.")


## 13. Read the training logs

`rollout/ep_rew_mean` is recent training reward, not unseen-track skill. `ep_len_mean` counts policy decisions, not raw simulator steps. `time/fps` counts policy transitions and includes overhead; its meaning is not lap time.

`train/approx_kl` measures approximate policy change; persistent large changes warrant inspecting the learning rate and update count. `clip_fraction` measures how often ratios exceeded the clip range. `entropy_loss` is negative entropy under SB3's convention; it is not the same as crash rate. `value_loss` measures critic target error. `explained_variance` near 1 suggests better value-target prediction; negative values suggest a poor predictor, but none of these metrics alone proves good driving. `train/std` summarizes learned noise-weight scales for gSDE; it is not the effective action standard deviation at every observation.

After validation starts, watch `validation/penalized_seconds` and `validation/completion_rate`. Plot training and held-out results together to detect a generalization gap. TensorBoard logs below are local to this runtime; the next cell copies them to Drive after training.


In [ ]:
# TensorBoard opens an inline dashboard. Run this after some training logs exist.
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR


In [ ]:
# Back up this session's logs after a training segment.
import shutil
if USE_DRIVE:
    shutil.copytree(LOG_DIR, RUN_DIR / "tensorboard_logs", dirs_exist_ok=True)
    print("Logs copied to Drive")
else:
    print("Local logs:", LOG_DIR)


## 14. Compare the current fine-tune against the protected baseline

Run after the additional training. Both models are evaluated on all three public tracks and validation seeds 10000-10009. Lower penalized time is better, but compare completion too. Public improvement with worse held-out results is a warning about specialization; neither result proves mystery-track performance. Keep the original if the copy is not a clear practical improvement. This cell does not automatically declare a winner or export it.


In [ ]:
evaluation_model = model
print("Evaluating current fine-tune:", evaluation_model.num_timesteps)
RUN_FINE_TUNE_EVALUATION = True
if RUN_FINE_TUNE_EVALUATION and model.num_timesteps > BASE_STEPS:
    baseline = json.loads(BASELINE_PATH.read_text())
    if baseline["parent"] != PARENT:
        raise ValueError("Baseline report belongs to a different parent")
    public_rows = evaluate_tracks(model, PUBLIC_SEEDS, split="public")
    validation_rows = evaluate_tracks(model, VALIDATION_SEEDS[:10], split="validation")
    report = {
        "timesteps": model.num_timesteps, "parent": PARENT, "sampling": SAMPLING, "context": EVALUATION_CONTEXT,
        "public": summarize(public_rows), "validation": summarize(validation_rows),
        "tracks": public_rows + validation_rows,
    }
    report["public_penalized_time_change"] = report["public"]["mean_penalized_seconds"] - baseline["public"]["mean_penalized_seconds"]
    report["validation_penalized_time_change"] = report["validation"]["mean_penalized_seconds"] - baseline["validation"]["mean_penalized_seconds"]
    (RUN_DIR / f"finetune_audit_{model.num_timesteps}.json").write_text(json.dumps(report, indent=2))
    print("BASELINE:", json.dumps({k: baseline[k] for k in ("public", "validation")}, indent=2))
    print("FINE-TUNE:", json.dumps({k: v for k, v in report.items() if k not in ("tracks", "parent", "sampling")}, indent=2))
    print("Negative time changes mean improvement. Keep the protected parent as fallback.")
else:
    print("No additional training yet, or audit disabled. Train section 12 before judging the copy.")


## 15. Watch the driver inside the notebook

Set `RECORD_VIDEO=True` to record one validation track. Colab displays an MP4 below; it does not open a desktop game window. The video is streamed to a local file to keep memory use modest, then copied to the run folder. Frame timing is for visualization; scoring comes from raw simulator steps.

After long training, rerun the checkpoint-selection cell above before this cell so `evaluation_model` represents the checkpoint you want. An early model may spin, brake or fail to accelerate. Inspect observations/action distributions and allow appropriate training before blaming the network size.


In [ ]:
RECORD_VIDEO = False
if RECORD_VIDEO:
    local_video = Path("/content") / f"{ARCHITECTURE}_preview.mp4" if USE_DRIVE else RUN_DIR / "preview.mp4"
    video_rows = evaluate_tracks(
        evaluation_model, (VALIDATION_SEEDS[0],),
        split="validation", video_path=local_video,
    )
    print(video_rows[0])
    if USE_DRIVE:
        shutil.copy2(local_video, RUN_DIR / "preview.mp4")
    display(Video(str(local_video), embed=True))


## 16. Final test and public tracks: run after choosing the model

Leave `RUN_FINAL_TEST=False` until the architecture/settings/checkpoint have been selected using validation. This uses all 20 untouched test tracks. If you change the model based on these results, the set is no longer an untouched test.

Public tracks are reported separately, with three deterministic runs each to match the challenge's repetition count. Fixed seeds plus a deterministic policy can produce identical repetitions, so this is not nine independent generalization examples. Compare the organizer's exact timing/loader rules before submission.


In [ ]:
RUN_FINAL_TEST = False
RUN_PUBLIC_TRACKS = False
if RUN_FINAL_TEST:
    rows = evaluate_tracks(evaluation_model, TEST_SEEDS, split="test")
    report = {"timesteps": evaluation_model.num_timesteps, "summary": summarize(rows), "tracks": rows}
    (RUN_DIR / "final_test.json").write_text(json.dumps(report, indent=2))
    print("Final test:", report["summary"])
if RUN_PUBLIC_TRACKS:
    repeated_seeds = tuple(seed for seed in PUBLIC_SEEDS for _ in range(3))
    rows = evaluate_tracks(evaluation_model, repeated_seeds, split="public")
    (RUN_DIR / "public_tracks.json").write_text(json.dumps(rows, indent=2))
    for seed in PUBLIC_SEEDS:
        print(seed, summarize([row for row in rows if row["track_seed"] == seed]))


## 17. Export weights and understand what has been saved

`latest.zip` contains the resumable SB3 training checkpoint. `best_validation.zip` is the selected checkpoint from periodic validation. Keep both. The cell below also exports a `.pth` bundle containing the selected policy's CPU tensors, its settings and its timestep count.

**The `.pth` bundle is not a complete official submission adapter.** Reconstructing this SB3 policy still requires the encoder class and the exact preprocessing, frame history reset, Gaussian-mean action handling and repeat timing. The organizer's loader interface is not supplied here. Do not submit an unexplained weights file and assume it will run. Saving a TorchScript model or a standalone actor later must reproduce this policy's deterministic actions across an entire rollout.

No pretrained CarRacing weights are downloaded. These files contain only the policy you train in this notebook.


In [ ]:
EXPORT_WEIGHTS = False
if EXPORT_WEIGHTS:
    weights_path = RUN_DIR / "selected_policy_weights.pth"
    torch.save({
        "policy_state_dict": {k: v.detach().cpu() for k, v in evaluation_model.policy.state_dict().items()},
        "config": CONFIG,
        "num_timesteps": evaluation_model.num_timesteps,
        "sb3_version": str(sb3.__version__),
        "gymnasium_version": str(gym.__version__),
    }, weights_path)
    print("Weights bundle:", weights_path)
print("Run folder:", RUN_DIR)
print("Checkpoints:", [path.name for path in RUN_DIR.glob("*.zip")])


## 18. Resume or keep the original

After restart, rerun installation/imports and Drive setup, keep this experiment folder, define wrappers/encoder, set `RESUME_FROM=str(RUN_DIR / "latest.zip")` in section 9, then run definitions and the remaining additional budget. Keep `RUN_SMOKE=False`. Baseline evaluation is cached by parent identity; you do not need to repeat it unchanged.

The original checkpoint was copied as `parent_400000.zip` and its source folder was never written. To watch the baseline again, set `evaluation_model=base_model` before the video cell. To watch the copy, set `evaluation_model=model`. Export only after choosing between these actual policies. Public fine-tuning is permitted according to your clarification, but should not be claimed as a guaranteed improvement or an untouched public test.


## References and limits of the architecture choice

- [PPO paper](https://arxiv.org/abs/1707.06347): clipped on-policy actor-critic learning; not a CarRacing winning guarantee.
- [SB3 PPO documentation](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html): the actual PPO implementation used here.
- [SB3 custom feature extractors](https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html): how our CNN is connected to the policy.
- [Gymnasium CarRacing](https://gymnasium.farama.org/environments/box2d/car_racing/): observations, actions and reward.
- [IMPALA paper](https://arxiv.org/abs/1802.01561): source of the residual encoder family; this notebook does not use the IMPALA learning algorithm.
- [Procgen paper](https://cdn.openai.com/procgen.pdf): motivation for varied levels and separate unseen evaluation. Its results do not prove that 100 CarRacing tracks are sufficient or that this encoder wins this race.
- [Colab FAQ](https://research.google.com/colaboratory/faq.html): hardware and session limits vary.

The notebook is an explainable baseline experiment. There is no measured claim here that either architecture is the fastest. Run the same protocol and let held-out results decide.
